# 02 - MLP Baseline para Classificação de Doenças em Plantas

## Perceptron Multicamadas com Backpropagation

**Objetivo:** Treinar MLP simples como baseline para comparação com CNN

**Arquitetura:**
- Entrada: Imagens 64×64 achatadas (4.096 features)
- Camada oculta 1: 256 neurônios + ReLU + Dropout(0.3)
- Camada oculta 2: 128 neurônios + ReLU + Dropout(0.3)
- Saída: Softmax para 10 classes
- Loss: CrossEntropy
- Otimizador: Adam

## Setup Inicial

In [ ]:
import sys
sys.path.insert(0, '/content/trabalho-rna-agro') if '/content/' in str(sys.path) else None

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# Importar módulos do projeto
from src.models import MLPBaseline
from src.data_loader import PlantDiseaseDataset
from src.training import Trainer
from src.utils import (
    set_seed, get_device, plot_training_history,
    plot_confusion_matrix, print_classification_report
)

# Configurações
set_seed(42)
device = get_device()
print(f"Device: {device}")

# Criar diretórios
Path("results/plots").mkdir(parents=True, exist_ok=True)
Path("results/confusion_matrices").mkdir(parents=True, exist_ok=True)

## 1. Carregar Dados (MLP usa 64x64)

In [ ]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Transformações para MLP (imagens 64x64)
img_size = 64

train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Criar datasets (assumindo que estão em data/train, data/val, data/test)
print("Carregando datasets...")

try:
    train_dataset = PlantDiseaseDataset(
        "data/train", split="train", img_size=img_size, transform=train_transform
    )
    val_dataset = PlantDiseaseDataset(
        "data/val", split="val", img_size=img_size, transform=val_transform
    )
    test_dataset = PlantDiseaseDataset(
        "data/test", split="test", img_size=img_size, transform=val_transform
    )
    
    num_classes = len(train_dataset.classes)
    print(f"✅ Datasets carregados!")
    print(f"  Treino: {len(train_dataset)} imagens")
    print(f"  Validação: {len(val_dataset)} imagens")
    print(f"  Teste: {len(test_dataset)} imagens")
    print(f"  Classes: {num_classes}")
    print(f"  Nomes: {train_dataset.classes}")
    
except Exception as e:
    print(f"❌ Erro ao carregar datasets: {e}")
    print("Execute primeiro: python scripts/prepare_datasets.py")

In [ ]:
# Criar DataLoaders
batch_size = 32
num_workers = 0  # Para Colab

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
)

print(f"DataLoaders criados com batch_size={batch_size}")

## 2. Definir e Treinar MLP

In [ ]:
# Criar modelo MLP
print("Criando modelo MLP...\n")

model = MLPBaseline(
    input_size=img_size * img_size,
    hidden_size=256,
    num_classes=num_classes
)

print(model)

# Contar parâmetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nParâmetros totais: {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")

In [ ]:
# Treinar modelo
print("Inicializando Trainer...\n")

trainer = Trainer(
    model=model,
    device=device,
    lr=0.001,
    optimizer_name="adam"
)

# Treinar por 50 épocas
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=50,
    save_best=True,
    save_dir="results"
)

## 3. Visualizar Histórico de Treinamento

In [ ]:
# Plotar histórico
fig = plot_training_history(
    history,
    save_path="results/plots/02_mlp_training_history.png"
)
plt.show()

print("✅ Gráfico salvo em: results/plots/02_mlp_training_history.png")

## 4. Avaliar no Conjunto de Teste

In [ ]:
# Carregar melhor modelo
best_model = MLPBaseline(
    input_size=img_size * img_size,
    hidden_size=256,
    num_classes=num_classes
)
best_model.load_state_dict(torch.load("results/best_model.pth", map_location=device))
best_model = best_model.to(device)

# Fazer predições
trainer.model = best_model
y_pred, y_conf, y_true = trainer.predict(test_loader)

print(f"Predições feitas: {len(y_pred)}")

In [ ]:
# Relatório de classificação
print_classification_report(
    y_true,
    y_pred,
    class_names=train_dataset.classes
)

## 5. Matriz de Confusão

In [ ]:
# Plotar matriz de confusão
fig = plot_confusion_matrix(
    y_true,
    y_pred,
    class_names=train_dataset.classes,
    save_path="results/confusion_matrices/02_mlp_confusion_matrix.png"
)
plt.show()

print("✅ Matriz de confusão salva em: results/confusion_matrices/02_mlp_confusion_matrix.png")

## 6. Resumo Final

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    RESUMO DO TREINAMENTO MLP                        ║
╚══════════════════════════════════════════════════════════════════════╝

ARQUITETURA:
  - Entrada: 64×64 → 4.096 features
  - Camada 1: 4.096 → 256 (ReLU + Dropout)
  - Camada 2: 256 → 128 (ReLU + Dropout)
  - Saída: 128 → num_classes
  - Total de parâmetros: ~1,3 milhão

RESULTADO:
  - Melhor acurácia de validação: {:.4f}
  - Acurácia em teste: {:.4f}
  
OBSERVAÇÕES:
  ✓ MLP é MUITO mais lento que CNN
  ✓ Acurácia é geralmente menor que CNN
  ✗ Não captura bem estruturas espaciais
  ✓ Útil como baseline para comparação

PRÓXIMO:
  Execute o notebook 03_cnn_transfer_learning.ipynb
  para ver o contraste com CNN (MobileNetV2)
""".format(
    max(history['val_acc']),
    (y_pred == y_true).mean()
))